In [1]:
using Pkg
Pkg.activate("../../../")
using Revise, QuantumStates, OpticalBlochEquations, UnitsToValue, DifferentialEquations, Plots, PlotlyJS, Statistics, LinearAlgebra, StaticArrays, DataFrames, CSV, ProgressMeter
using StaticArrays, RectiGrids, StatsBase
RESULTS_PATH = "RESULTS"

  Activating project at `~/Documents/Works/MIT/RaX/Simu/Molecule-Sims`


"RESULTS"

In [ ]:
#################################################################################################
## Helper Functions and Constants

# Function to get the basis element with the maximum coefficient (highest probability amplitude)
# This function is crucial for identifying the most probable quantum state after solving the Hamiltonian.
function get_max_coeff_basis_element(state)
    # Compute the probability for each basis state by taking the square of the absolute value of its coefficient.
    # This step is important as it relates to the probability distribution over quantum states as discussed in the paper.
    coeffs = abs.(state.coeffs) .^ 2
    # Find the index corresponding to the maximum probability.
    # The paper suggests that the molecular states of interest are often dominated by a single quantum state,
    # making this method a practical approach to identify the dominant state.
    max_idx = argmax(coeffs)
    # Return the basis element (specific quantum state) corresponding to the maximum probability.
    return state.basis[max_idx]
end

# Function to extract quantum numbers from a quantum state
# This function allows us to retrieve specific quantum numbers (N, J, F, M) from the most probable basis element of a state.
function extract_quantum_numbers_from_state(state)
    # Get the basis element with the highest probability amplitude, which represents the most likely quantum state.
    basis_elem = get_max_coeff_basis_element(state)
    # Extract the relevant quantum numbers (N: rotational, J: total angular momentum, F: total angular momentum with nuclear spin, M: projection of F)
    # These quantum numbers are critical for identifying and manipulating states in the laser cooling process, as described in the paper.
    quantum_numbers = Dict(:N => basis_elem.N, :J => basis_elem.J, :F => basis_elem.F, :M => basis_elem.M)
    return quantum_numbers
end

# Function to find a specific quantum state within a list of states based on given quantum numbers
# This function is useful when we need to locate a state that matches certain quantum criteria.
function find_state_by_quantum_numbers(states, N, J, F, M)
    # Loop through the list of states to compare their quantum numbers with the desired values.
    for state in states
        # Extract the quantum numbers from the current state.
        qn = extract_quantum_numbers_from_state(state)
        # If the quantum numbers match the desired values, return the corresponding state.
        if qn[:N] == N && qn[:J] == J && qn[:F] == F && qn[:M] == M
            return state
        end
    end
    # If no matching state is found, throw an error.
    # This ensures that the simulation only proceeds with valid and identifiable quantum states, as discussed in the paper.
    error("State with quantum numbers N=$N, J=$J, F=$F, M=$M not found.")
end

# Function to retrieve all information about the most probable basis element of a state
# This is particularly useful for understanding the detailed structure of the state, which is necessary for the precise laser interactions described in the paper.
function all_state_info(state)
    # Get the basis element with the highest coefficient, which is the most likely quantum state.
    basis_elem = get_max_coeff_basis_element(state)
    return basis_elem
end

# Constants used throughout the simulation, based on the experimental parameters discussed in the paper.
_μB = (μB / h) * 1e-4  # Bohr magneton in units compatible with the Hamiltonian (Hz/Gauss)
λ = 552e-9  # Wavelength of the cooling laser, as specified in the paper (in meters)
Γ = 2π * 5.7e6  # Natural linewidth of the YbF molecular transition, derived from the paper's experimental setup (in Hz)
m = @with_unit 191 "u"  # Mass of the YbF molecule, consistent with the values used in the paper (in atomic mass units)
k = 2π / λ  # Wave number corresponding to the laser wavelength, a key quantity in the momentum transfer during laser cooling

#################################################################################################
## Setting up the states

# Function to define the quantum number bounds for the X (ground) and A (excited) electronic states
# This setup is critical for constructing the molecular basis states, as discussed in the paper's theoretical model.
function define_QN_bounds()
    # Quantum number bounds for the X state (ground electronic state)
    # These bounds are chosen based on the rotational and hyperfine structure of the X state, as detailed in the paper.
    QN_bounds_X = (
        S=1 / 2,  # Electron spin quantum number, S=1/2 for YbF in the ground state
        I=1 / 2,  # Nuclear spin quantum number, I=1/2 for fluorine in YbF
        Λ=0,  # Projection of the electronic orbital angular momentum along the molecular axis (Λ=0 for the X state)
        N=0:3  # Range of rotational quantum numbers considered, based on the typical energy levels involved in the cooling process
    )

    # Quantum number bounds for the A state (excited electronic state)
    # The bounds here account for both rotational and electronic excitations, consistent with the Hund's case A model.
    QN_bounds_A = (
        S=1 / 2,  # Electron spin quantum number, S=1/2 in the A state as well
        I=1 / 2,  # Nuclear spin quantum number, same as in the X state
        Λ=(-1, 1),  # Possible projections of the electronic orbital angular momentum (Λ = ±1 for the A state)
        J=1/2:3/2  # Range of total angular momentum quantum numbers, taking into account the fine structure splitting
    )

    # Alternative quantum number bounds for the A state, allowing for additional configurations
    # These bounds are relevant when considering transitions involving different rotational levels in the excited state.
    QN_bounds_A2 = (
        S = 1/2,  # Electron spin quantum number, consistent across all configurations
        I = 1/2,  # Nuclear spin quantum number, consistent across all configurations
        Λ = (-1,1),  # Range of Λ values, reflecting possible electronic orbital angular momentum projections
        N = 0:3  # Range of rotational quantum numbers, similar to the X state
    )

    # Return the defined bounds, which will be used to construct the basis states for the Hamiltonians.
    return QN_bounds_X, QN_bounds_A, QN_bounds_A2
end

# Function to generate the Hamiltonians for both the X and A electronic states
# The Hamiltonians incorporate various interactions as described in the paper, including rotational, spin-rotation, and hyperfine interactions.
function generate_hamiltonians(B)
    # Retrieve the quantum number bounds for the X and A states, which define the space of possible quantum states.
    QN_bounds_X, QN_bounds_A, QN_bounds_A2 = define_QN_bounds()

    # Generate the basis states for the X state (Hund's case B) using the specified quantum number bounds.
    X_state_basis = enumerate_states(HundsCaseB_LinearMolecule, QN_bounds_X)
    # Define the Hamiltonian operator for the X state, which includes rotational, spin-rotation, and hyperfine interactions.
    # The specific terms correspond to the molecular constants discussed in the paper.
    X_state_operator = :(
        BX * Rotation +  # Rotational energy term, BX is the rotational constant
        DX * RotationDistortion +  # Correction for centrifugal distortion, DX is the distortion constant
        γX * SpinRotation +  # Spin-rotation coupling, γX is the spin-rotation constant
        bFX * Hyperfine_IS +  # Hyperfine interaction term, bFX is the Fermi contact term
        cX * (Hyperfine_Dipolar / 3)  # Dipolar hyperfine interaction, cX is the dipolar constant
    )

    # Define the specific parameter values for the X state Hamiltonian
    # These parameters are based on experimental data and theoretical calculations provided in the paper.
    X_state_parameters = QuantumStates.@params begin
        BX = 7233.8271e6  # Rotational constant (in Hz)
        DX = 0.0  # Rotational distortion constant (assumed negligible)
        γX = -13.41679e6  # Spin-rotation constant (in Hz)
        bFX = 170.26374e6  # Fermi contact term (in Hz)
        cX = 85.4028e6  # Dipolar hyperfine constant (in Hz)
    end

    # Create the Hamiltonian object for the X state using the defined basis, operator, and parameters.
    X_state_ham = Hamiltonian(basis=X_state_basis, operator=X_state_operator, parameters=X_state_parameters)

    # Define the Zeeman interaction components for the magnetic field in x, y, and z directions
    # These components model the interaction of the molecule's magnetic moment with an external magnetic field.
    # The paper discusses the significance of the Zeeman effect in the manipulation of quantum states.
    Zeeman_x(state, state′) = (Zeeman(state, state′, -1) - Zeeman(state, state′, 1)) / sqrt(2)
    Zeeman_y(state, state′) = im * (Zeeman(state, state′, -1) + Zeeman(state, state′, 1)) / sqrt(2)
    Zeeman_z(state, state′) = Zeeman(state, state′, 0)

    # Add the Zeeman interactions to the X state Hamiltonian, allowing for the inclusion of magnetic field effects.
    # The paper highlights how magnetic fields are used to manipulate the molecular states during the experiment.
    X_state_ham = add_to_H(X_state_ham, :B_x, (gS * _μB * 1e-6) * Zeeman_x)
    X_state_ham = add_to_H(X_state_ham, :B_y, (gS * _μB * 1e-6) * Zeeman_y)
    X_state_ham = add_to_H(X_state_ham, :B_z, (gS * _μB * 1e-6) * Zeeman_z)
    # Set the magnetic field components for the Hamiltonian
    # B_x is set to zero, while B_y and B_z are set to the same value, representing a field at 45 degrees to the molecular axis.
    X_state_ham.parameters.B_x = 0.0
    X_state_ham.parameters.B_y = B / sqrt(2)
    X_state_ham.parameters.B_z = B / sqrt(2)

    # Solve the Hamiltonian to obtain the energy levels (eigenstates) for the X state.
    # This step is essential for identifying the quantum states that will participate in the laser cooling process.
    evaluate!(X_state_ham)
    QuantumStates.solve!(X_state_ham)
    # Extract the ground states with N=1, which are the specific rotational levels of interest for the cooling transitions.
    all_grounds = X_state_ham.states[5:16]
    ground_states = [state for state in all_grounds if extract_quantum_numbers_from_state(state)[:N] == 1]

    # Generate the basis states for the A state (Hund's case A) using the specified quantum number bounds.
    A_state_basis = enumerate_states(HundsCaseA_LinearMolecule, QN_bounds_A)

    # Define the Hamiltonian operator for the A state, which includes electronic, rotational, spin-orbit, and hyperfine interactions.
    # These interactions are crucial for understanding the dynamics of the excited state, as described in the paper.
    A_state_operator = :(
        T_A * DiagonalOperator +  # Electronic energy term, T_A is the energy of the A state relative to the X state
        Be_A * Rotation +  # Rotational energy term, Be_A is the rotational constant for the A state
        Aso_A * SpinOrbit +  # Spin-orbit coupling term, Aso_A is the spin-orbit constant
        q_A * ΛDoubling_q +  # Lambda doubling term (q parameter), representing the splitting of Λ levels
        p_A * ΛDoubling_p2q +  # Lambda doubling term (p parameter), representing higher-order splitting
        a * Hyperfine_IL +  # Hyperfine interaction term (I · L), a is the hyperfine constant for nuclear-electronic interaction
        d * Hyperfine_Dipolar_d  # Dipolar hyperfine interaction term, d is the dipolar constant
    )

    # Define the specific parameter values for the A state Hamiltonian
    # These parameters are consistent with the experimental and theoretical values provided in the paper.
    A_state_parameters = QuantumStates.@params begin
        T_A = 542.8102 * 10^12  # Electronic energy difference (in Hz)
        Be_A = 40.9304 * 10^12  # Rotational constant for the A state (in Hz)
        Aso_A = 7.4276 * 10^9  # Spin-orbit coupling constant (in Hz)
        p_A = 11.882 * 10^9  # Lambda doubling constant (p parameter) (in Hz)
        q_A = 0  # Lambda doubling constant (q parameter) (in Hz)
        a = -0.8e6  # Hyperfine interaction constant (in Hz)
        d = -4.6e6  # Dipolar hyperfine constant (in Hz)
    end

    # Create the Hamiltonian object for the A state using the defined basis, operator, and parameters.
    A_state_ham = Hamiltonian(basis=A_state_basis, operator=A_state_operator, parameters=A_state_parameters)
    # Solve the Hamiltonian to obtain the energy levels (eigenstates) for the A state.
    # This is critical for identifying the excited states that will be populated by laser transitions.
    evaluate!(A_state_ham)
    QuantumStates.solve!(A_state_ham)

    # Extract the excited states corresponding to the lowest energy levels in the A state.
    excited_states = A_state_ham.states[1:4]
    A_state_basis = enumerate_states(HundsCaseB_LinearMolecule, QN_bounds_A2)
    excited_states = convert_basis(excited_states, A_state_basis)

    # Return the ground states, excited states, and the Hamiltonians and basis for both states.
    # These are the key inputs for simulating the optical transitions described in the paper.
    return ground_states, excited_states, X_state_ham, A_state_ham, A_state_basis
end

# Function to generate Transition Dipole Moments (TDMs) between the X and A states
# TDMs are critical for determining the strength of the optical transitions, as discussed in the paper's section on laser cooling.
function generate_tdms(X_state_ham, ground_states, excited_states, A_state_basis)
    # Initialize a zero matrix to store transition dipole moments (TDMs) for all states.
    # This matrix will be filled with the computed TDMs between the ground and excited states.
    d = zeros(ComplexF64, 16, 16, 3)
    # Initialize a smaller matrix to store TDMs specifically between the ground and excited states.
    d_ge = zeros(ComplexF64, 12, 4, 3)
    # Calculate the TDMs between different bases (X state and A state).
    # The TDMs are calculated based on the dipole operator, which governs the interaction between the molecule and the laser field.
    basis_tdms = get_tdms_two_bases(X_state_ham.basis, A_state_basis, TDM)
    # Compute the TDMs between the ground and excited states and store them in the smaller matrix.
    tdms_between_states!(d_ge, basis_tdms, ground_states, excited_states)
    # Insert the computed TDMs into the appropriate submatrix of the larger matrix.
    d[1:12, 13:16, :] .= d_ge
    return d
end

#################################################################################################
## Setting up the lasers

# Function to define the laser fields and their parameters
# The paper discusses the importance of laser configurations in achieving efficient cooling, and this function sets up those configurations.
function generate_lasers(config, pol_J12, ω_J12s, s_J12_sidebands)
    # Define a function to return a constant polarization value, representing the electric field vector of the laser.
    # The polarization of the laser is crucial for selecting specific transitions, as discussed in the paper.
    ϵ(ϵ_val) = t -> ϵ_val
    use_single_state = false
    # Generate a list of polarizations for the laser fields, all set to the input polarization (pol_J12).
    polarizations = [pol_J12 for _ in ω_J12s]

    if use_single_state
        # Define the wave vectors (k̂) and polarizations (ϵ) for the lasers in the XY‖ configuration.
        # This configuration refers to lasers aligned parallel to the molecular plane, as discussed in the experimental setup.
        k̂ = +x̂; ϵ1 = ϵ(rotate_pol(pol_J12, ẑ)); laser1 = Field(k̂, ϵ1, ω_J12, s_J12)
        k̂ = -x̂; ϵ2 = ϵ(rotate_pol(pol_J12, -ẑ)); laser2 = Field(k̂, ϵ2, ω_J12, s_J12)
        k̂ = +ŷ; ϵ3 = ϵ(rotate_pol(pol_J12, ẑ)); laser3 = Field(k̂, ϵ3, ω_J12, s_J12)
        k̂ = -ŷ; ϵ4 = ϵ(rotate_pol(pol_J12, -ẑ)); laser4 = Field(k̂, ϵ4, ω_J12, s_J12)

        # Combine all lasers into a list for the XY‖ configuration.
        lasers_XY_parallel = [laser1, laser2, laser3, laser4]

        # Define the wave vectors (k̂) and polarizations (ϵ) for the lasers in the XY⊥ configuration.
        # This configuration refers to lasers aligned perpendicular to the molecular plane.
        k̂ = +x̂; ϵ1 = ϵ(rotate_pol(pol_J12, ŷ)); laser1 = Field(k̂, ϵ1, ω_J12, s_J12)
        k̂ = -x̂; ϵ2 = ϵ(rotate_pol(pol_J12, -ŷ)); laser2 = Field(k̂, ϵ2, ω_J12, s_J12)
        k̂ = +ŷ; ϵ3 = ϵ(rotate_pol(pol_J12, ẑ)); laser3 = Field(k̂, ϵ3, ω_J12, s_J12)
        k̂ = -ŷ; ϵ4 = ϵ(rotate_pol(pol_J12, -ẑ)); laser4 = Field(k̂, ϵ4, ω_J12, s_J12)

        # Combine all lasers into a list for the XY⊥ configuration.
        lasers_XY_perpendicular = [laser1, laser2, laser3, laser4]
    else
        # Generate laser fields for the XY‖ configuration without using a single state.
        lasers_plus_x = [Field(+x̂, ϵ, ω, s) for (ϵ, j, ω, s) in zip([ϵ(rotate_pol(pol_J12, ẑ)) for j in 1:length(ω_J12s)], polarizations, ω_J12s, s_J12_sidebands)]
        lasers_minus_x = [Field(-x̂, ϵ, ω, s) for (ϵ, j, ω, s) in zip([ϵ(rotate_pol(pol_J12, -ẑ)) for j in 1:length(ω_J12s)], polarizations, ω_J12s, s_J12_sidebands)]
        lasers_plus_y = [Field(+ŷ, ϵ, ω, s) for (ϵ, j, ω, s) in zip([ϵ(rotate_pol(pol_J12, ẑ)) for j in 1:length(ω_J12s)], polarizations, ω_J12s, s_J12_sidebands)]
        lasers_minus_y = [Field(-ŷ, ϵ, ω, s) for (ϵ, j, ω, s) in zip([ϵ(rotate_pol(pol_J12, -ẑ)) for j in 1:length(ω_J12s)], polarizations, ω_J12s, s_J12_sidebands)]

        # Combine all lasers into lists for the XY‖ and XY⊥ configurations.
        lasers_XY_parallel = [lasers_plus_x; lasers_minus_x; lasers_plus_y; lasers_minus_y]

        laser_plus_x = [Field(+x̂, ϵ, ω, s) for (ϵ, j, ω, s) in zip([ϵ(rotate_pol(pol_J12, ŷ)) for j in 1:length(ω_J12s)], polarizations, ω_J12s, s_J12_sidebands)]
        laser_minus_x = [Field(-x̂, ϵ, ω, s) for (ϵ, j, ω, s) in zip([ϵ(rotate_pol(pol_J12, -ŷ)) for j in 1:length(ω_J12s)], polarizations, ω_J12s, s_J12_sidebands)]
        laser_plus_y = [Field(+ŷ, ϵ, ω, s) for (ϵ, j, ω, s) in zip([ϵ(rotate_pol(pol_J12, ẑ)) for j in 1:length(ω_J12s)], polarizations, ω_J12s, s_J12_sidebands)]
        laser_minus_y = [Field(-ŷ, ϵ, ω, s) for (ϵ, j, ω, s) in zip([ϵ(rotate_pol(pol_J12, -ẑ)) for j in 1:length(ω_J12s)], polarizations, ω_J12s, s_J12_sidebands)]
        lasers_XY_perpendicular = [laser_plus_x; laser_minus_x; laser_plus_y; laser_minus_y]
    end
    # Return the generated laser configurations, which are crucial for the cooling process described in the paper.
    return lasers_XY_parallel, lasers_XY_perpendicular
end

#################################################################################################
## Define and solve Optical Bloch Equations (OBE)

# Function to solve the Optical Bloch Equations (OBE) for the system
# The OBE describes the time evolution of the quantum states under the influence of laser fields, as discussed in the paper's theoretical model.
function solve_obe(states, lasers, d)
    # Initialize the density matrix ρ0 with the ground state fully populated.
    # The density matrix represents the probability distribution over quantum states.
    ρ0 = zeros(ComplexF64, length(states), length(states))
    particle = OpticalBlochEquations.Particle()
    ρ0[1,1] = 1.0  # Set initial population to the ground state

    # Define a function to update the probabilities during the simulation, based on the velocity and position grid.
    # This function is essential for simulating the cooling process as the molecule interacts with the laser fields.
    function prob_func!(prob, scan_values_grid, i)
        p = prob.p
        # Update the particle's velocity and position based on the current grid values.
        p.v .= (0, scan_values_grid[i].v, 0)
        p.v .= round_vel(p.v, p.freq_res)
        p.r0 .= scan_values_grid[i].r
        return prob
    end

    # Define a function to extract the output from the solution, specifically the forces acting on the molecule.
    # The force is a key quantity in the cooling process, as it determines how the molecule's velocity is reduced.
    function output_func(p, sol)
        f = p.force_last_period
        return (f[1], f[2], f[3])  # Return the force components
    end

    # Set the frequency resolution for the OBE solver, which determines the accuracy of the simulation.
    freq_res = 1e-1
    # Set up the OBE problem with the initial conditions, particle parameters, and laser-molecule interactions.
    # This setup is directly related to the paper's discussion on modeling the dynamics of the cooling process.
    p = obe(ρ0, particle, states, lasers, d, true, true; λ=λ, Γ=Γ, freq_res=freq_res)

    # Define the time span for the simulation, which should cover multiple periods of the particle's motion.
    t_end = 5p.period+1; tspan = (0., t_end)
    # Solve the OBE using an ODE solver, which integrates the equations over time.
    # This step is computationally intensive and crucial for predicting the behavior of the system under laser cooling.
    @time prob = ODEProblem(ρ!, p.ρ0_vec, tspan, p, reltol=1e-3, save_on=false)

    # Define the ranges for scanning the forces and populations across different velocities and positions.
    di = 2
    rs = vcat([(n1/(di+1), n2/(di+1), n3/(di+1)) .* 2π for n1 ∈ 0:di, n2 ∈ 0:di, n3 ∈ 0:di]...)
    vs = 0.:.5:6

    # Generate a grid for scanning the parameters, which allows us to explore the system's behavior across a range of conditions.
    scan_values = (r = rs, v = vs)
    scan_values_grid = RectiGrids.grid(scan_values)
    # Perform the force and population scan by solving the OBE at each grid point.
    # This provides a detailed map of how the cooling forces and state populations evolve under the influence of the lasers.
    forces, populations = force_scan_v2(prob, scan_values_grid, prob_func!, output_func)
    return forces, populations
end

# Function to calculate laser frequencies (omegas) for transitions between the ground and excited states
# These frequencies are directly related to the detuning and linewidth discussed in the paper.
function get_omegas(ground_states, excited_states, detunning, Γ)
    # Filter the ground states of interest, specifically those with all vibrational quantum numbers set to zero.
    # These states are most relevant for the cooling transitions, as they are the lowest energy states.
    g_interest_states = [state for state in ground_states if (all_state_info(state).v_1 == 0 && all_state_info(state).v_2 == 0 && all_state_info(state).v_3 == 0)][1:4]
    # Get the energies of the ground states, which will be used to calculate the transition frequencies.
    input_energies = [energy(st) for st in g_interest_states]

    # Get the energies of the excited states and sort them by energy.
    # This allows us to identify the specific transitions that will be driven by the lasers.
    target_states = sort(excited_states, by = energy)
    target_energy = [energy(st) for st in excited_states][1]
    A_energy = target_energy  # Use the energy of the first excited state

    # Calculate the detuned laser frequencies based on the energy difference between ground and excited states.
    # The detuning is a key parameter in controlling the cooling process, as described in the paper.
    δJ12 = detunning * Γ
    ω_J12s = [2π * (A_energy - inp_e) + δJ12 for inp_e in input_energies]
    return ω_J12s
end

# Function to save the simulation results as a DataFrame and write them to a CSV file
# This step is important for post-processing and analyzing the results, as discussed in the paper's data analysis section.
function save_as_df(forces, populations, saving_path)
    # Initialize an array to store the averaged forces across different velocities.
    # Averaging helps to smooth out fluctuations and provides a clearer picture of the cooling effect.
    averaged_forces = []
    for (i, v) ∈ enumerate(vs)
        # Find the indices corresponding to the current velocity in the scan grid.
        idxs = [j for (j, x) ∈ enumerate(scan_values_grid) if x.v == v]
        # Filter out unrealistic force values (those exceeding a threshold) and compute the mean force for this velocity.
        _forces = [f[1] for f in forces[idxs] if abs(f[1]) <= 1e3]
        push!(averaged_forces, mean([f[1] for f in _forces]))
    end
    # Create a DataFrame with the velocities, forces, and state populations, which can be used for further analysis.
    df = DataFrame(v=vs, force=averaged_forces, population=populations)
    # Write the DataFrame to a CSV file, making the simulation data accessible for plotting and interpretation.
    CSV.write(saving_path, df)
end

# Main function to run the entire simulation
# This function orchestrates the process by running the OBE solver for various detunings and magnetic field configurations.
# The results are saved for each set of parameters, allowing for a comprehensive study of the cooling process as described in the paper.
function run_simulation(detunings, magnetic_fields, CONFIGURATION)
    # Define the natural linewidth of the transition, a key parameter in the cooling process.
    Γ = 2π * 5.7e6
    # Loop over all magnetic field strengths and detuning values, exploring a wide range of experimental conditions.
    for B in magnetic_fields
        for detuning in detunings
            @show B, detuning  # Display the current parameters being simulated, for monitoring progress
            # Generate the Hamiltonians and states for the given magnetic field
            ground_states, excited_states, X_state_ham, A_state_ham, A_state_basis = generate_hamiltonians(10.0 * 1e-4)
            # Generate the transition dipole moments (TDMs) for the current states
            d = generate_tdms(X_state_ham, ground_states, excited_states, A_state_basis)
            # Calculate the laser frequencies for the given detuning
            ω_J12s = get_omegas(ground_states, excited_states, detuning, Γ)
            # Define a function to set the laser intensity saturation
            s_func(s) = (r,t) -> s
            I_sat =  π*h*c*Γ/(3λ^3) / 10  # Calculate the saturation intensity
            s_J12 = s_func(100.0)  # Set the laser intensity scaling factor
            ratios = [1, 1/2, 1/2, 0]  # Define sideband ratios, which are used to account for laser sidebands
            s_J12_sidebands = [s_func(ratio * 100.0) for ratio in ratios]
            # Generate the laser fields for the specified configuration (XY_parallel or XY_perpendicular)
            lasers_XY_parallel, lasers_XY_perpendicular = generate_lasers(CONFIGURATION, σ⁺, ω_J12s, s_J12_sidebands)

            # Select the laser configuration to use in the simulation
            if CONFIGURATION == "XY_parallel"
                lasers = lasers_XY_parallel
            else
                lasers = lasers_XY_perpendicular
            end

            # Combine the ground and excited states into a single array, representing the complete set of states in the simulation
            states = [ground_states; excited_states]
            # Solve the Optical Bloch Equations (OBE) for the given states and laser fields
            forces, populations = solve_obe(states, lasers, d)
            # Save the simulation results (forces and populations) to a CSV file for later analysis
            save_as_df(forces, populations, "data/$(CONFIGURATION)_B_$(B)_detuning_$(detuning).csv")
        end
    end
end


In [ ]:
detunings = [1.,]# 3, 6, 9, 12]
magnetic_fields = [0.0, 1.0, 5.0, 25.0] .* 1e-4
magnetic_fields = magnetic_fields[2:2]
run_simulation(detunings, magnetic_fields, "XY_parallel")
